# Target Population Weighting and Bayesian Nonparametric Survival Inference

In this notebook, we estimate target-population survival effects using reconstructed individual patient data (IPD) augmented with synthetic baseline covariates.

The goal is to move beyond naive pooled survival comparisons by defining a target population through auxiliary summary information and reweighting the pooled data accordingly.

This notebook:
1. loads the reconstructed IPD with covariates
2. loads auxiliary summaries and benchmark pooled estimates
3. defines a target population
4. computes target-population weights
5. estimates weighted survival effects
6. quantifies uncertainty using a Bayesian nonparametric weighting scheme

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from src.causal.estimands import pooled_survival_difference
from src.causal.pooling import (
    compute_covariate_means,
    reweight_to_target,
    weighted_survival_difference,
)

# Reproducibility
np.random.seed(733)

# Paths
DATA_DIR = Path("../data/processed")

# Load processed inputs
df_cov = pd.read_csv(DATA_DIR / "real_kmgpt_ipd_with_covariates.csv")
combined_aux = pd.read_csv(DATA_DIR / "combined_auxiliary_summaries.csv")
naive_pooled = pd.read_csv(DATA_DIR / "naive_pooled_effect.csv")

print("Loaded df_cov:", df_cov.shape)
print("Loaded combined_aux:", combined_aux.shape)
print("Loaded naive_pooled:", naive_pooled.shape)

df_cov.head()

Loaded df_cov: (776, 18)
Loaded combined_aux: (4, 7)
Loaded naive_pooled: (1, 4)


,trial_id,subgroup,time,event,arm_label,curve,treatment,age_median,male_rate,ecog0_rate,metastatic_rate,biomarker_rate,age,male,ecog0,metastatic,biomarker,stage
0,KEYNOTE-181,high_pdl1,0.731707,1,Chemotherapy,0,0,63.0,0.869,0.401,0.924,0.7,56.369120,1,0,0,0,1.165542
1,KEYNOTE-181,high_pdl1,0.890244,0,Chemotherapy,0,0,63.0,0.869,0.401,0.924,0.7,68.148584,1,1,1,0,1.822921
2,KEYNOTE-181,high_pdl1,1.048780,1,Chemotherapy,0,0,63.0,0.869,0.401,0.924,0.7,58.035384,1,0,1,1,2.330231
3,KEYNOTE-181,high_pdl1,1.243902,1,Chemotherapy,0,0,63.0,0.869,0.401,0.924,0.7,61.334523,1,1,1,1,1.968789
4,KEYNOTE-181,high_pdl1,1.365854,1,Chemotherapy,0,0,63.0,0.869,0.401,0.924,0.7,73.194077,1,0,1,1,2.342307


## Covariates Available for Weighting

The augmented dataset now contains individual-level survival outcomes, treatment assignment, and synthetic baseline covariates informed by trial-level auxiliary summaries.

We next select a small set of covariates to define the target population and construct weighting constraints.

In [2]:
# Inspect available columns and choose weighting covariates

print(df_cov.columns.tolist())

covariates = ["age", "male", "ecog0", "metastatic", "biomarker", "stage"]
t0 = 12.0

print("\nChosen covariates for weighting:", covariates)
print("Evaluation time t0:", t0)

df_cov[covariates].describe()

['trial_id', 'subgroup', 'time', 'event', 'arm_label', 'curve', 'treatment', 'age_median', 'male_rate', 'ecog0_rate', 'metastatic_rate', 'biomarker_rate', 'age', 'male', 'ecog0', 'metastatic', 'biomarker', 'stage']

Chosen covariates for weighting: ['age', 'male', 'ecog0', 'metastatic', 'biomarker', 'stage']
Evaluation time t0: 12.0


,age,male,ecog0,metastatic,biomarker,stage
count,776.000000,776.000000,776.000000,776.000000,776.000000,776.000000
mean,62.987504,0.864691,0.394330,0.796392,0.640464,2.211855
std,8.248887,0.342274,0.489021,0.402941,0.480174,0.450939
min,35.105425,0.000000,0.000000,0.000000,0.000000,0.827052
25%,57.270466,1.000000,0.000000,1.000000,0.000000,1.902646
50%,62.837263,1.000000,0.000000,1.000000,1.000000,2.243770
75%,68.331990,1.000000,1.000000,1.000000,1.000000,2.554033
max,89.357448,1.000000,1.000000,1.000000,1.000000,3.247082


## Target Population Definition

We define a target population using covariate summaries. This represents the population for which we want to estimate the survival effect.

The target is specified through moments of baseline covariates rather than individual-level data, reflecting settings where only aggregate information is available.

This target is slightly healthier than pooled data and has lower biomarker average. It should be different enough to induce reweighting

In [3]:
target_means = pd.Series({
    "age": 60.0,
    "male": 0.85,
    "ecog0": 0.30,
    "metastatic": 0.75,
    "biomarker": 0.60,
    "stage": 2.0,
})

target_means

age           60.00
male           0.85
ecog0          0.30
metastatic     0.75
biomarker      0.60
stage          2.00
dtype: float64

## Target-Population Weighting

We construct weights so that the weighted sample matches the target covariate moments.

This defines a target-population estimand:
$$
\Delta(t_0) = S_1^{(target)}(t_0) - S_0^{(target)}(t_0)
$$

We also examine the stability of the weights through effective sample size (ESS) and compare weighted covariate means to the target.

In [4]:
# Compute weights
w_target = reweight_to_target(
    df=df_cov,
    covariates=covariates,
    target_means=target_means,
)

# Basic diagnostics
print("Min weight:", w_target.min())
print("Max weight:", w_target.max())

# Effective sample size (ESS)
w_norm = w_target / w_target.sum()
ess = 1.0 / np.sum(w_norm**2)

print("ESS:", ess)

Min weight: 1.1677411038876052e-05
Max weight: 0.015405880553413126
ESS: 328.00576231898236


/Users/jonathanma/Desktop/MSEnotes/733_NPBayes/FinalProject/NPBS-project-MaZhu/src/causal/pooling.py:129: RuntimeWarning: divide by zero encountered in matmul
  logits = X @ theta #softmax stabilization
/Users/jonathanma/Desktop/MSEnotes/733_NPBayes/FinalProject/NPBS-project-MaZhu/src/causal/pooling.py:129: RuntimeWarning: overflow encountered in matmul
  logits = X @ theta #softmax stabilization
/Users/jonathanma/Desktop/MSEnotes/733_NPBayes/FinalProject/NPBS-project-MaZhu/src/causal/pooling.py:129: RuntimeWarning: invalid value encountered in matmul
  logits = X @ theta #softmax stabilization


In [5]:
weighted_means = compute_covariate_means(
    df_cov,
    covariates,
    weights=w_target,
)

comparison = pd.DataFrame({
    "target": target_means,
    "weighted": weighted_means,
})

comparison

,target,weighted
age,60.00,60.000811
male,0.85,0.850017
ecog0,0.30,0.300332
metastatic,0.75,0.749632
biomarker,0.60,0.599900
stage,2.00,2.000498


## Target-Population Weighting Diagnostics

The weighting procedure successfully matches the target covariate moments:

- All weighted covariate means closely align with the specified target values.
- This confirms that the exponential tilting procedure is functioning as intended.

The effective sample size (ESS) is approximately 328, compared to the original sample size of 776.

This reduction in ESS indicates moderate reweighting, reflecting a meaningful shift from the pooled population to the target population. The weights are somewhat concentrated, but not excessively so, suggesting that the target population remains reasonably supported by the observed data.

In [6]:
weighted_est = weighted_survival_difference(
    df=df_cov,
    weights=w_target,
    t0=t0,
    random_state=733,
)

weighted_est

{'t0': 12.0,
 'S0': 0.2610120778249844,
 'S1': 0.4951238754225493,
 'Delta': 0.2341117975975649}

## Target-Population Survival Effect

After reweighting to the target population, the estimated survival probabilities at $t_0 = 12$ are:

- $S_0(t_0) = 0.261$
- $S_1(t_0) = 0.495$
- $\Delta(t_0) = 0.234$

Compared to the naive pooled estimate ($\Delta \approx 0.169$), the target-population effect is substantially larger.

This indicates that the treatment effect is sensitive to the underlying population, and that the target population places more weight on individuals for whom immunotherapy is relatively more effective. The difference between naive and weighted estimates highlights the importance of accounting for cross-trial heterogeneity when defining causal survival effects.

We can see a roughly 38% increase in pooled delta. Because the target population is worse health, lower biomarker, and younger, we can infer the weighting is not uniform, so certain subpopulations benefit more from immunotherapy. treatment effect is NOT invariant across populations.

## Bayesian Nonparametric Uncertainty

To quantify uncertainty in the weighted estimate, we place a Dirichlet distribution over the weights centered at the target weights.

This produces a posterior distribution over the survival contrast $\Delta(t_0)$, reflecting uncertainty due to finite sample size and reweighting.

In [7]:
rng = np.random.default_rng(733)

alpha = 200
n_draws = 300

delta_draws = []

for i in range(n_draws):
    w_draw = rng.dirichlet(alpha * w_target)

    est = weighted_survival_difference(
        df=df_cov,
        weights=w_draw,
        t0=t0,
        random_state=733 + i,
    )

    delta_draws.append(est["Delta"])

delta_draws = np.array(delta_draws)

In [8]:
summary = {
    "mean": delta_draws.mean(),
    "sd": delta_draws.std(),
    "q025": np.quantile(delta_draws, 0.025),
    "q975": np.quantile(delta_draws, 0.975),
}

summary

{'mean': np.float64(0.2050476115732194),
 'sd': np.float64(0.07481539265739884),
 'q025': np.float64(0.05169459792302812),
 'q975': np.float64(0.3606609406933556)}

## Target-Population Survival Effect with BNP Uncertainty

Using target-population weighting and Bayesian nonparametric uncertainty quantification, the estimated survival contrast at $t_0 = 12$ is:

- Posterior mean: $\Delta(t_0) = 0.205$
- Posterior standard deviation: $0.075$
- 95% credible interval: $(0.052, 0.361)$

The posterior mean is larger than the naive pooled estimate ($\Delta \approx 0.169$), indicating that the treatment effect depends on the target population. The credible interval excludes zero, suggesting a consistent survival benefit of immunotherapy in the target population.

The relatively wide credible interval reflects uncertainty induced by both finite sample size and reweighting, particularly given the moderate effective sample size after weighting.

## Output Saving

In [ ]:
pd.DataFrame({
    "weight": w_target
}).to_csv("../data/processed/target_weights.csv", index=False)

pd.DataFrame([weighted_est]).to_csv(
    "../data/processed/weighted_estimate.csv",
    index=False
)

pd.DataFrame({
    "delta": delta_draws
}).to_csv("../data/processed/bnp_delta_draws.csv", index=False)

pd.DataFrame([summary]).to_csv(
    "../data/processed/bnp_summary.csv",
    index=False
)

pd.DataFrame({
    "ESS": [ess],
    "min_weight": [w_target.min()],
    "max_weight": [w_target.max()],
}).to_csv("../data/processed/weight_diagnostics.csv", index=False)

comparison.to_csv(
    "../data/processed/target_vs_weighted_covariates.csv"
)